In [0]:
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=3/2025/02/26/00_26_56/*.parquet")

In [0]:
from pyspark.sql.functions import col, count, month, year, to_date, concat, lit, when, last_day, lpad, sum as _sum

# Filter for US & jobs posted after 2017-01-01
df_us = df.filter((col('nation_name') == "United States") & (col('posted') >= "2017-01-01") & (col('posted') < "2019-01-01"))

# Filter to only include naics code 21 for grey industry jobs
df_us = df_us.filter((col('naics2') == 21))

# Compile the DataFrame with necessary transformations
new_df = (
    df_us.withColumn("month", lpad(month(col("posted")).cast("string"), 2, "0"))  # Ensures 01, 02, ..., 12
    .withColumn("year", year(col("posted")))
    .groupBy("year", "month", "laa_admin_area_2", "min_years_experience", "isco_08_1_name")
    .agg(
        count(when(col("employment_type") == 1, True)).alias("full_time_count"),
        count(when(col("employment_type") == 2, True)).alias("part_time_count"),
        count("*").alias("total_count")
    )
    .withColumnRenamed("laa_admin_area_2", "county_id")
    .withColumnRenamed("isco_08_1_name", "career_type")
    .withColumn("start_date", to_date(concat(col("year"), lit("-"), col("month"), lit("-01")), "yyyy-MM-dd"))
    .withColumn("end_date", last_day(col("start_date")))  # Ensure correct last day
)

# Display the final compiled DataFrame
display(new_df)

year,month,county_id,min_years_experience,career_type,full_time_count,part_time_count,total_count,start_date,end_date
2017,07,US56005,1,Craft and Related Trades Workers,1,0,1,2017-07-01,2017-07-31
2017,09,US12086,5,Professionals,6,0,6,2017-09-01,2017-09-30
2017,09,US6073,5,Professionals,7,0,7,2017-09-01,2017-09-30
2017,09,US29051,4,Professionals,2,0,2,2017-09-01,2017-09-30
2018,05,US4013,null,"Plant and Machine Operators, and Assemblers",27,0,27,2018-05-01,2018-05-31
2018,05,US42081,null,"Plant and Machine Operators, and Assemblers",0,0,3,2018-05-01,2018-05-31
2018,06,US40133,null,"Plant and Machine Operators, and Assemblers",1,0,1,2018-06-01,2018-06-30
2018,06,US48085,10,Professionals,3,0,3,2018-06-01,2018-06-30
2018,03,US29189,10,Managers,3,0,3,2018-03-01,2018-03-31
2017,10,US1101,null,Professionals,3,0,3,2017-10-01,2017-10-31


In [0]:
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/grey_vacancies_data_2017_18"

# Append data as CSV (instead of Parquet)
new_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(output_path)